### Assignment 1 - Text Mining - Text Categorization
#### Imports:

In [3]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import precision_recall_fscore_support

#### Task 1:
Addressing all 20 categories

In [4]:
# categories=None loads all 20 newsgroups
remove = ("headers", "footers", "quotes")

data_train = fetch_20newsgroups(subset="train", categories=None, shuffle=True,random_state=42, remove=remove)
data_test = fetch_20newsgroups(subset="test", categories=None, shuffle=True, random_state=42, remove=remove)

y_train = data_train.target
y_test = data_test.target
target_names = data_train.target_names

print("number of categories:", len(target_names))
print("training documents:", len(data_train.data))
print("test documents:", len(data_test.data))
print(target_names)

number of categories: 20
training documents: 11314
test documents: 7532
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


#### Task 2:
Comparing 3 classifiers on

In [5]:
# helper that trains a classifier and prints macro precision, recall and F1
def run_classifier(clf, X_train, X_test):
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="macro", zero_division=0)
    return precision, recall, f1

classifiers = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Linear SVC", LinearSVC()),
    ("Complement NB", ComplementNB()),
]

# quick comparison with tf-idf features
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(data_train.data)
X_test = vectorizer.transform(data_test.data)

for name, clf in classifiers:
    p, r, f = run_classifier(clf, X_train, X_test)
    print(f"{name}: precision={p:.3f} recall={r:.3f} F1={f:.3f}")

Logistic Regression: precision=0.677 recall=0.661 F1=0.659
Linear SVC: precision=0.688 recall=0.681 F1=0.681
Complement NB: precision=0.719 recall=0.697 F1=0.693


#### Task 4:
Counts vs tf vs tf-idf

In [6]:
feature_types = [
    ("counts", CountVectorizer()),
    ("tf", TfidfVectorizer(use_idf=False)),
    ("tf-idf", TfidfVectorizer(use_idf=True)),
]

rows = []
for feat_name, vectorizer in feature_types:
    X_train = vectorizer.fit_transform(data_train.data)
    X_test = vectorizer.transform(data_test.data)
    for clf_name, clf in classifiers:
        p, r, f = run_classifier(clf, X_train, X_test)
        rows.append([clf_name, feat_name, p, r, f])
        print(f"done: {clf_name} + {feat_name}")

results = pd.DataFrame(rows, columns=["classifier", "features", "precision", "recall", "f1"])
print(results.round(3))

# one table per metric, for the report
for metric in ["precision", "recall", "f1"]:
    print("\n", metric)
    print(results.pivot(index="classifier", columns="features", values=metric).round(3))

# best combination, to use in Task 4
best_row = results.loc[results["f1"].idxmax()]
print("\nbest:", best_row["classifier"], "with", best_row["features"], "- F1 =", round(best_row["f1"], 3))

done: Logistic Regression + counts


/Users/alex/PycharmProjects/TextMining1/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


done: Linear SVC + counts
done: Complement NB + counts
done: Logistic Regression + tf
done: Linear SVC + tf
done: Complement NB + tf
done: Logistic Regression + tf-idf
done: Linear SVC + tf-idf
done: Complement NB + tf-idf
            classifier features  precision  recall     f1
0  Logistic Regression   counts      0.595   0.587  0.588
1           Linear SVC   counts      0.570   0.562  0.564
2        Complement NB   counts      0.668   0.653  0.637
3  Logistic Regression       tf      0.554   0.546  0.540
4           Linear SVC       tf      0.659   0.652  0.651
5        Complement NB       tf      0.705   0.674  0.668
6  Logistic Regression   tf-idf      0.677   0.661  0.659
7           Linear SVC   tf-idf      0.688   0.681  0.681
8        Complement NB   tf-idf      0.719   0.697  0.693

 precision
features             counts     tf  tf-idf
classifier                                
Complement NB         0.668  0.705   0.719
Linear SVC            0.570  0.659   0.688
Logistic Regr

#### Task 4:


#### Task 5: